In [ ]:
import os
import shutil


def download_stockfish():
    stockfish_dir = "/kaggle/working/stockfish/"
    stockfish_binary = os.path.join(stockfish_dir, "stockfish")
    source_path = "/kaggle/input/stockfish/other/binary/1/stockfish"

    os.makedirs(stockfish_dir, exist_ok=True)

    try:
        if not os.path.exists(stockfish_binary):
            shutil.copy2(source_path, stockfish_binary)
            os.chmod(stockfish_binary, 0o755)
        return stockfish_binary
    except Exception as e:
        print(f"Error setting up Stockfish: {e}")
        return None


STOCKFISH_PATH = download_stockfish()

In [ ]:
from dataclasses import dataclass


@dataclass
class TrainingConfig:
    hidden_size: int = 256
    intermediate_size: int = 1024
    num_hidden_layers: int = 6
    num_attention_heads: int = 8
    max_position_embeddings: int = 512
    max_length: int = 512

    batch_size: int = 96
    learning_rate: float = 3e-4
    weight_decay: float = 0.01

    train_epochs: int = 6

    max_games: int = 4096 + 2048
    train_split: float = 0.9

    stockfish_path: str = ""
    stockfish_depth: int = 10
    stockfish_temperature: float = 50.0
    invalid_move_loss_weight: float = 0.5

    log_every_n_batches: int = 50

    save_dir: str = "chess_model"


config = TrainingConfig()
config.stockfish_path = STOCKFISH_PATH  # type: ignore

In [ ]:
from typing import List, Optional
import json


class ChessTokenizer:
    def __init__(self):
        self.special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]
        self.vocab = self._build_vocab()
        self.token_to_id = {token: idx for idx, token in enumerate(self.vocab)}
        self.id_to_token = {idx: token for token, idx in self.token_to_id.items()}

        self.pad_token = "<PAD>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"
        self.unk_token = "<UNK>"

        self.pad_token_id = self.token_to_id[self.pad_token]
        self.bos_token_id = self.token_to_id[self.bos_token]
        self.eos_token_id = self.token_to_id[self.eos_token]
        self.unk_token_id = self.token_to_id[self.unk_token]

    def _build_vocab(self) -> List[str]:
        vocab = []
        vocab.extend(self.special_tokens)

        files = "abcdefgh"
        ranks = "12345678"
        squares = [f + r for f in files for r in ranks]

        for from_sq in squares:
            for to_sq in squares:
                vocab.append(f"{from_sq}{to_sq}")

                if from_sq[1] == "7" and to_sq[1] == "8":
                    for piece in ["q", "r", "b", "n"]:
                        vocab.append(f"{from_sq}{to_sq}{piece}")

                if from_sq[1] == "2" and to_sq[1] == "1":
                    for piece in ["q", "r", "b", "n"]:
                        vocab.append(f"{from_sq}{to_sq}{piece}")

        return vocab

    def encode_move(self, move_uci: str) -> str:
        return move_uci

    def decode_move(self, move_token: str) -> Optional[str]:
        if move_token in self.token_to_id and len(move_token) >= 4:
            return move_token
        return None

    def encode(self, moves: List[str]) -> List[int]:
        tokens = [self.bos_token]
        tokens.extend(moves)
        tokens.append(self.eos_token)

        return [self.token_to_id.get(token, self.unk_token_id) for token in tokens]

    def decode(self, token_ids: List[int]) -> List[str]:
        return [self.id_to_token.get(id, self.unk_token) for id in token_ids]

    def __len__(self):
        return len(self.vocab)

    def save_pretrained(self, path: str):
        os.makedirs(path, exist_ok=True)

        vocab_data = {
            "vocab": self.vocab,
            "special_tokens": self.special_tokens,
        }

        with open(os.path.join(path, "vocab.json"), "w") as f:
            json.dump(vocab_data, f, indent=2)

        config = {
            "pad_token": self.pad_token,
            "bos_token": self.bos_token,
            "eos_token": self.eos_token,
            "unk_token": self.unk_token,
        }

        with open(os.path.join(path, "tokenizer_config.json"), "w") as f:
            json.dump(config, f, indent=2)


tokenizer = ChessTokenizer()

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Example encoding: {tokenizer.encode(['e2e4', 'e7e5'])}")

In [ ]:
from torch.utils.data import Dataset
from typing import Dict
import torch


class ChessDataset(Dataset):
    def __init__(
        self, samples: List[Dict], tokenizer: ChessTokenizer, max_length: int = 512
    ):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        moves = sample["moves"]

        input_ids = self.tokenizer.encode(moves)

        if len(input_ids) > self.max_length:
            input_ids = input_ids[: self.max_length]

        input_ids = input_ids + [self.tokenizer.pad_token_id] * (
            self.max_length - len(input_ids)
        )

        attention_mask = [
            1 if id != self.tokenizer.pad_token_id else 0 for id in input_ids
        ]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }


def collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "attention_mask": torch.stack([item["attention_mask"] for item in batch]),
    }

In [ ]:
from typing import Tuple
from torch.utils.data import DataLoader
import random


def create_dataloaders(
    samples: List[Dict],
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
) -> Tuple[DataLoader, DataLoader]:
    random.shuffle(samples)
    split_idx = int(config.train_split * len(samples))
    train_samples = samples[:split_idx]
    val_samples = samples[split_idx:]

    print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

    train_dataset = ChessDataset(train_samples, tokenizer, config.max_length)
    val_dataset = ChessDataset(val_samples, tokenizer, config.max_length)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=collate_fn,
    )

    return train_loader, val_loader


print("DataLoader creation functions defined")

In [ ]:
import torch.nn as nn
from transformers import LlamaConfig, LlamaForCausalLM


class ChessTransformer(nn.Module):
    def __init__(self, config: LlamaConfig, vocab_size: int):
        super().__init__()

        config.vocab_size = vocab_size
        self.config = config

        self.model = LlamaForCausalLM(config)

    def forward(self, input_ids, attention_mask=None, labels=None):
        if labels is not None:
            outputs = self.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            return {"loss": outputs.loss, "logits": outputs.logits}
        else:
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            return {"loss": None, "logits": outputs.logits}

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from typing import List, Optional, Dict

import chess
import chess.pgn  # type: ignore
import chess.engine

import torch
import multiprocessing
import pickle


def _evaluate_position(args):
    stockfish_path, board_fen, depth, legal_moves_uci = args

    try:
        engine = chess.engine.SimpleEngine.popen_uci(stockfish_path)
        engine.configure({"Threads": 1})

        board = chess.Board(board_fen)
        scores = {}

        for move_uci in legal_moves_uci:
            temp_board = board.copy()
            move = chess.Move.from_uci(move_uci)
            temp_board.push(move)

            try:
                info = engine.analyse(
                    temp_board, chess.engine.Limit(depth=depth), multipv=1
                )
                score = info["score"].relative.score(mate_score=10000)
                scores[move_uci] = -score
            except:
                scores[move_uci] = 0

        engine.quit()
        return board_fen, scores
    except:
        return board_fen, {}


class StockfishEvaluator:
    def __init__(
        self,
        stockfish_path: str,
        depth: int = 5,
        num_workers: int = 0,
    ):
        self.stockfish_path = stockfish_path
        self.depth = depth
        self.num_workers = num_workers or multiprocessing.cpu_count()
        self.evaluations = {}
        self.pending_positions = {}
        self.executor = ProcessPoolExecutor(max_workers=self.num_workers)

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.executor:
            self.executor.shutdown(wait=True)

    def save_evaluations(self, path: str):
        with open(path, "wb") as f:
            pickle.dump(self.evaluations, f)
        print(f"Saved {len(self.evaluations)} evaluations to {path}")

    def load_evaluations(self, path: str):
        with open(path, "rb") as f:
            self.evaluations = pickle.load(f)
        print(f"Loaded {len(self.evaluations)} evaluations from {path}")

    def reconstruct_board(
        self, token_ids: List[int], tokenizer
    ) -> Optional[chess.Board]:
        board = chess.Board()
        tokens = tokenizer.decode(token_ids)

        for token in tokens:
            if token in ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]:
                continue
            try:
                move = chess.Move.from_uci(token)
                if move in board.legal_moves:
                    board.push(move)
                else:
                    return None
            except:
                return None
        return board

    def get_stockfish_target(
        self,
        input_ids: torch.Tensor,
        position_idx: int,
        tokenizer,
    ) -> Optional[int]:
        board = self.reconstruct_board(input_ids[:position_idx].tolist(), tokenizer)

        if board is None or board.is_game_over():
            return None

        position_fen = board.fen()

        if position_fen in self.evaluations:
            move_scores = self.evaluations[position_fen]
        elif position_fen in self.pending_positions:
            return None
        else:
            legal_moves_uci = [move.uci() for move in board.legal_moves]
            future = self.executor.submit(
                _evaluate_position,
                (self.stockfish_path, position_fen, self.depth, legal_moves_uci),
            )
            self.pending_positions[position_fen] = future
            return None

        if not move_scores:
            return None

        best_move = max(move_scores.items(), key=lambda x: x[1])[0]
        best_token_id = tokenizer.token_to_id.get(best_move, tokenizer.unk_token_id)

        return best_token_id

    def collect_results(self):
        completed_positions = []

        for position_fen, future in list(self.pending_positions.items()):
            if future.done():
                try:
                    fen, scores = future.result(timeout=0)
                    if scores:
                        self.evaluations[fen] = scores
                    completed_positions.append(position_fen)
                except:
                    completed_positions.append(position_fen)

        for fen in completed_positions:
            del self.pending_positions[fen]

        return len(completed_positions)

In [ ]:
import pandas as pd
import io


def load_dataset(csv_path, max_games=None):
    print(f"Loading dataset from {csv_path}...")
    df = pd.read_csv(csv_path)

    print(f"Total games in CSV: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")

    if max_games:
        df = df.head(max_games)
        print(f"Limited to {len(df)} games")

    return df


def process_games(df: pd.DataFrame, max_games: Optional[int] = None) -> List[Dict]:
    samples = []

    games_to_process = len(df) if max_games is None else min(max_games, len(df))

    print(f"Processing {games_to_process} games...")

    for idx in range(games_to_process):
        if idx % 1000 == 0:
            print(f"Processed {idx}/{games_to_process} games")

        try:
            pgn_text = df.iloc[idx]["pgn"]

            game = chess.pgn.read_game(io.StringIO(pgn_text))
            if game is None:
                continue

            moves = [move.uci() for move in game.mainline_moves()]

            if len(moves) >= 10:
                samples.append({"moves": moves})

        except Exception as e:
            continue

    print(f"Extracted {len(samples)} games.")
    return samples

In [ ]:
def create_model(
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
    device: torch.device,
) -> ChessTransformer:

    model_config = LlamaConfig(
        vocab_size=len(tokenizer),
        hidden_size=config.hidden_size,
        intermediate_size=config.intermediate_size,
        num_hidden_layers=config.num_hidden_layers,
        num_attention_heads=config.num_attention_heads,
        max_position_embeddings=config.max_position_embeddings,
        rms_norm_eps=1e-5,
        initializer_range=0.02,
        use_cache=False,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    model = ChessTransformer(model_config, len(tokenizer))

    model = model.to(device)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    return model


def save_checkpoint(
    model: ChessTransformer,
    tokenizer: ChessTokenizer,
    config: TrainingConfig,
    epoch: int,
    metrics: Dict[str, float],
):
    save_path = os.path.join(config.save_dir, f"epoch_{epoch}")
    os.makedirs(save_path, exist_ok=True)

    model_to_save = model.module if isinstance(model, nn.DataParallel) else model

    model_to_save.model.save_pretrained(save_path, safe_serialization=True)

    tokenizer.save_pretrained(save_path)

    with open(os.path.join(save_path, "training_metadata.json"), "w") as f:
        json.dump(
            {
                "epoch": epoch,
                "metrics": metrics,
            },
            f,
            indent=2,
        )

    print(f"Checkpoint saved to {save_path}")

In [ ]:
from tqdm import tqdm
import torch
import torch.nn.functional as F
import random


def train_epoch(
    model,
    dataloader,
    optimizer,
    device,
    tokenizer,
    stockfish_evaluator,
    prefetch_batches=5,
):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_predictions = 0
    total_positions = 0
    total_cache_hits = 0

    batch_buffer = []

    print("Pre-fetching initial batches...")
    prefetch_bar = tqdm(
        enumerate(dataloader), total=prefetch_batches, desc="Prefetching"
    )
    for batch_idx, batch in prefetch_bar:
        if batch_idx >= prefetch_batches:
            break

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_size = input_ids.size(0)

        for i in range(batch_size):
            seq_length = attention_mask[i].sum().item()
            if seq_length > 2:
                pos = random.randint(1, seq_length - 2)
                stockfish_evaluator.get_stockfish_target(input_ids[i], pos, tokenizer)

        batch_buffer.append(batch)

        if batch_idx % 2 == 0:
            _ = stockfish_evaluator.collect_results()
            prefetch_bar.set_postfix(
                {
                    "cached": len(stockfish_evaluator.evaluations),
                    "pending": len(stockfish_evaluator.pending_positions),
                }
            )

    print("Collecting initial evaluations...")
    while len(stockfish_evaluator.pending_positions) > 50:
        _ = stockfish_evaluator.collect_results()

    print(
        f"Starting training with {len(stockfish_evaluator.evaluations)} cached positions"
    )

    progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Training")

    for batch_idx, batch in progress_bar:
        stockfish_evaluator.collect_results()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad()

        batch_size = input_ids.size(0)

        targets = []
        final_valid_indices = []
        final_valid_positions = []

        for i in range(batch_size):
            seq_length = attention_mask[i].sum().item()
            if seq_length > 2:
                pos = random.randint(1, seq_length - 2)

                target_token_id = stockfish_evaluator.get_stockfish_target(
                    input_ids[i], pos, tokenizer
                )

                total_positions += 1

                if target_token_id is not None:
                    targets.append(target_token_id)
                    final_valid_indices.append(i)
                    final_valid_positions.append(pos)
                    total_cache_hits += 1

        if len(final_valid_indices) == 0:
            progress_bar.set_postfix(
                {
                    "loss": "N/A",
                    "acc": "N/A",
                    "hit": f"{total_cache_hits/max(total_positions, 1):.1%}",
                    "cached": len(stockfish_evaluator.evaluations),
                    "pending": len(stockfish_evaluator.pending_positions),
                }
            )
            continue

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        if isinstance(outputs, dict):
            logits = outputs["logits"]
        else:
            logits = outputs.logits

        if logits.dim() == 4:
            logits = logits.squeeze(0)

        selected_logits = []
        for i, (valid_idx, pos) in enumerate(
            zip(final_valid_indices, final_valid_positions)
        ):
            selected_logits.append(logits[valid_idx, pos, :])

        selected_logits = torch.stack(selected_logits)
        targets_tensor = torch.tensor(targets, dtype=torch.long, device=device)

        loss = F.cross_entropy(
            selected_logits.contiguous(),
            targets_tensor.contiguous(),
        )

        predicted = torch.argmax(selected_logits, dim=-1)
        batch_correct = (predicted == targets_tensor).sum().item()
        batch_predictions = len(targets)

        total_correct += batch_correct
        total_predictions += batch_predictions

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        cache_hit_rate = total_cache_hits / max(total_positions, 1)

        progress_bar.set_postfix(
            {
                "loss": f"{loss.item():.4f}",
                "acc": f"{batch_correct/max(batch_predictions, 1):.3f}",
                "hit": f"{cache_hit_rate:.1%}",
                "cached": len(stockfish_evaluator.evaluations),
            }
        )

    avg_loss = total_loss / len(dataloader) if total_loss > 0 else 0.0
    avg_accuracy = total_correct / total_predictions if total_predictions > 0 else 0.0
    cache_hit_rate = total_cache_hits / max(total_positions, 1)

    return {
        "loss": avg_loss,
        "accuracy": avg_accuracy,
        "total_predictions": total_predictions,
        "cache_hit_rate": cache_hit_rate,
    }


def evaluate(model, dataloader, device, tokenizer, stockfish_evaluator):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_predictions = 0

    progress_bar = tqdm(dataloader, desc="Evaluating")

    with torch.no_grad():
        for batch in progress_bar:
            stockfish_evaluator.collect_results()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            batch_size = input_ids.size(0)

            targets = []
            valid_indices = []
            valid_positions = []

            for i in range(batch_size):
                seq_length = attention_mask[i].sum().item()
                if seq_length > 2:
                    pos = random.randint(1, seq_length - 2)
                    target_token_id = stockfish_evaluator.get_stockfish_target(
                        input_ids[i], pos, tokenizer
                    )
                    if target_token_id is not None:
                        targets.append(target_token_id)
                        valid_indices.append(i)
                        valid_positions.append(pos)

            if len(valid_indices) > 0:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

                if isinstance(outputs, dict):
                    logits = outputs["logits"]
                else:
                    logits = outputs.logits

                if logits.dim() == 4:
                    logits = logits.squeeze(0)

                selected_logits = []
                for i, (valid_idx, pos) in enumerate(
                    zip(valid_indices, valid_positions)
                ):
                    selected_logits.append(logits[valid_idx, pos, :])

                selected_logits = torch.stack(selected_logits)
                targets_tensor = torch.tensor(targets, dtype=torch.long, device=device)

                loss = F.cross_entropy(
                    selected_logits.contiguous(), targets_tensor.contiguous()
                )

                predicted = torch.argmax(selected_logits, dim=-1)
                batch_correct = (predicted == targets_tensor).sum().item()
                batch_predictions = len(targets)

                total_loss += loss.item()
                total_correct += batch_correct
                total_predictions += batch_predictions

                progress_bar.set_postfix(
                    {
                        "loss": f"{loss.item():.4f}",
                        "acc": f"{batch_correct/batch_predictions:.3f}",
                    }
                )

    avg_loss = total_loss / len(dataloader) if total_loss > 0 else 0.0
    avg_accuracy = total_correct / total_predictions if total_predictions > 0 else 0.0

    return {
        "loss": avg_loss,
        "accuracy": avg_accuracy,
        "total_predictions": total_predictions,
    }

In [ ]:
df = load_dataset(
    csv_path="/kaggle/input/chesscom-user-games-60000-games/club_games_data.csv",
    max_games=config.max_games,
)
print(f"\nDataset shape: {df.shape}")
if "pgn" in df.columns:
    print(f"First game preview:\n{df['pgn'].iloc[0][:200]}...")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Available GPUs: {torch.cuda.device_count()}")

In [ ]:
print("\n" + "=" * 50)
print("TRAINING")
print("=" * 50)

samples = process_games(df, max_games=config.max_games)
train_loader, val_loader = create_dataloaders(samples, tokenizer, config)

model = create_model(tokenizer, config, device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)

eval_cache_path = "stockfish_evaluations.pkl"

with StockfishEvaluator(
    config.stockfish_path,
    config.stockfish_depth,
) as stockfish_evaluator:

    if os.path.exists(eval_cache_path):
        print(f"\nLoading cached evaluations from {eval_cache_path}")
        stockfish_evaluator.load_evaluations(eval_cache_path)

    for epoch in range(config.train_epochs):
        print(f"\nEpoch {epoch + 1}/{config.train_epochs}")
        print("-" * 50)

        train_metrics = train_epoch(
            model,
            train_loader,
            optimizer,
            device,
            tokenizer,
            stockfish_evaluator,
        )
        print(f"Train Loss: {train_metrics['loss']:.4f}")
        print(f"Train Accuracy: {train_metrics['accuracy']:.4f}")
        print(f"Train Positions: {train_metrics['total_predictions']}")
        print(f"Cache Hit Rate: {train_metrics['cache_hit_rate']:.1%}")
        print(f"Cached Positions: {len(stockfish_evaluator.evaluations)}")

        val_metrics = evaluate(
            model, val_loader, device, tokenizer, stockfish_evaluator
        )
        print(f"Val Loss: {val_metrics['loss']:.4f}")
        print(f"Val Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Val Positions: {val_metrics['total_predictions']}")

        save_checkpoint(model, tokenizer, config, epoch + 1, val_metrics)

        stockfish_evaluator.save_evaluations(eval_cache_path)